In [1]:
import pandas as pd
import numpy as np
import glob
import os
import gc

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

RAW_DIR = "../data/raw"

In [2]:
csv_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
print(f"Found {len(csv_files)} files:")
for f in csv_files:
    print(" -", os.path.basename(f))

dfs = []
for f in csv_files:
    df_part = pd.read_csv(f, encoding='latin1', low_memory=False)
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)

# free the list of individual dataframes now that they're merged
del dfs
gc.collect()

print(f"\nCombined shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Found 8 files:
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv

Combined shape: (2830743, 79)
Memory usage: 1.81 GB


In [3]:
df.columns = df.columns.str.strip()
df['Label'] = df['Label'].str.replace('ï¿½', '-', regex=False)
print(df['Label'].unique())

<ArrowStringArray>
[                    'BENIGN',                       'DDoS',
                   'PortScan',                        'Bot',
               'Infiltration',   'Web Attack - Brute Force',
           'Web Attack - XSS', 'Web Attack - Sql Injection',
                'FTP-Patator',                'SSH-Patator',
              'DoS slowloris',           'DoS Slowhttptest',
                   'DoS Hulk',              'DoS GoldenEye',
                 'Heartbleed']
Length: 15, dtype: str


In [4]:
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='float')

for col in df.select_dtypes(include=['int64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='integer')

print(f"Memory usage after downcasting: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Memory usage after downcasting: 0.92 GB


In [5]:
label_counts = df['Label'].value_counts()
print(label_counts)
print(f"\nBENIGN percentage: {label_counts['BENIGN'] / len(df) * 100:.2f}%")

Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack - Brute Force         1507
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

BENIGN percentage: 80.30%


In [6]:
print("NaN counts (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))

inf_counts = np.isinf(df.select_dtypes(include=[np.number])).sum()
print("\nInf counts (top 10):")
print(inf_counts.sort_values(ascending=False).head(10))

# replace inf with NaN, then drop affected rows (known small % of data)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
before = len(df)
df.dropna(subset=['Flow Bytes/s', 'Flow Packets/s'], inplace=True)
print(f"\nDropped {before - len(df)} rows with NaN/Inf in flow rate columns")

NaN counts (top 10):
Flow Bytes/s                   1358
Flow Duration                     0
Destination Port                  0
Total Backward Packets            0
Total Length of Fwd Packets       0
Total Length of Bwd Packets       0
Total Fwd Packets                 0
Fwd Packet Length Max             0
Fwd Packet Length Min             0
Fwd Packet Length Std             0
dtype: int64

Inf counts (top 10):
Flow Packets/s            2867
Flow Bytes/s              1509
Total Fwd Packets            0
Total Backward Packets       0
Destination Port             0
Flow Duration                0
Fwd Packet Length Max        0
Fwd Packet Length Min        0
Fwd Packet Length Mean       0
Fwd Packet Length Std        0
dtype: int64

Dropped 2867 rows with NaN/Inf in flow rate columns


In [7]:
before = len(df)
df.drop_duplicates(inplace=True)
gc.collect()
print(f"Dropped {before - len(df)} duplicate rows ({(before-len(df))/before*100:.2f}%)")
print(f"Final shape: {df.shape}")
print(f"Final memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Dropped 307084 duplicate rows (10.86%)
Final shape: (2520792, 79)
Final memory usage: 0.84 GB


In [8]:
df['Label_binary'] = df['Label'].apply(lambda x: 'BENIGN' if x == 'BENIGN' else 'ATTACK')
print(df['Label_binary'].value_counts())

rare_classes = ['Heartbleed', 'Infiltration', 'Web Attack - Sql Injection']
df = df[~df['Label'].isin(rare_classes)]
gc.collect()

print(f"\nMulticlass label distribution (rare classes excluded):")
print(df['Label'].value_counts())
print(f"\nFinal memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Label_binary
BENIGN    2095051
ATTACK     425741
Name: count, dtype: int64

Multiclass label distribution (rare classes excluded):
Label
BENIGN                      2095051
DoS Hulk                     172846
DDoS                         128014
PortScan                      90694
DoS GoldenEye                 10286
FTP-Patator                    5931
DoS slowloris                  5385
DoS Slowhttptest               5228
SSH-Patator                    3219
Bot                            1948
Web Attack - Brute Force       1470
Web Attack - XSS                652
Name: count, dtype: int64

Final memory usage: 0.88 GB


In [9]:
df.to_parquet("../data/processed/cleaned_flows.parquet", index=False)
print("Saved checkpoint.")

Saved checkpoint.
